In [ ]:
import numpy as np
import math
def general_combination(n,lambda_):
    S = 1
    for i in range(n):
        S *= (i+lambda_)
    return S

def gamma_ratio(d,k):#k从1开始
    if (d-k)/2 > 20:
        z = (d-k)/2
        omega = z - 1/4
        rho = 3/4
        B = [1, -rho/6, rho*(5*rho+1)/60, -rho*(35*rho**2+21*rho+4)/504, rho*(175*rho**3+210*rho**2+101*rho+18)/2160] 
        S = sum([ general_combination(2*j,-1/2)*B[j]*(omega**(1/2-2*j))/math.factorial(2*j)  for j in range(5)])
    else:
        S = math.gamma((d-k+1)/2)/math.gamma((d-k)/2)
    return S

def gamma_star(z):
    G = [1, 1/12, 1/288, -139/51840, -571/2488320, 163879/209018880, 5246819/75246796800]
    S = sum([ G[k]*z**(-k) for k in range(7)])
    return S
    

def gamma_square_ratio(d,k):#k从1开始
    if (d-k-1) > 25:
        lambda_ = (d-k-1)/2
        S = np.sqrt(lambda_/math.pi)*gamma_star(2*lambda_)/(gamma_star(lambda_)**2)
    else:
        lambda_ = (d-k-1)/2
        S = (2**(1-2*lambda_))*math.gamma(2*lambda_)/(math.gamma(lambda_)**2)
    return S
    
def computation_h_alpha(d):#(X_{d-1},\dots,X_1)
    constant = [np.array([1])]
    List = []
    constant.append(np.sqrt(d)*np.ones(d))
    L = []
    for k in range(1,d-1):
        List = np.sqrt(d*(d+2))*np.ones(d-k+1)
        List[0] = 1/np.sqrt((np.sqrt(math.pi)*(d-k)*((d-k)**2-1)*gamma_ratio(d,k)*gamma_square_ratio(d,k)/(d*(d+2))))
        L.append(np.array(List[:]))
    L.append(np.ones(2)*np.sqrt(d*(d+2))/2)
    constant.append(L[:])
    return constant
'''
        for i2 in range(i1,d):
            alpha_J = [ 0 for j in range(d)]
            alpha_J[i1-1] += 1
            alpha_J[i2-1] += 1
            alpha_top_J_plus1 = [ 0 for j in range(d)]
            for j in range(1,d+1):
                if j < i1 and j < i2:
                    alpha_top_J_plus1[j-1] = 2
                elif j < i1 or j < i2:
                    alpha_top_J_plus1[j-1] = 1
            lambda_J = [ alpha_top_J_plus1[j]+(d-j-2)/2 for j in range(d-1)]
            S = 1
            for j in range(d-2):
                S *= 2**(1-2*lambda_J[j])*math.gamma(alpha_J[j]+2*lambda_J[j])/(factorial(alpha_J[j])*(lambda_J[j]+alpha_J[j])*(math.gamma(lambda_J[j])**2))
            if i1 == (d-1) or i2 == (d-1):
                S *= coe
                List.append(1/np.sqrt(S))
            else:
                S *= 2*coe
                List.append(1/np.sqrt(S))
        List.append(List[-1])

'''

'''
    List = []
    for i2 in range(1,d):
        alpha_J = [ 0 for j in range(d)]
        alpha_J[i2-1] = 1
        alpha_J[-1] = 1
        alpha_top_J_plus1 = [ 2 for j in range(i2-1)]+ [1 for j in range(d-i2)]
        lambda_J = [ alpha_top_J_plus1[j]+(d-j-2)/2 for j in range(d-1)]
        S = 1
        for j in range(d-2):
            S *= factorial(alpha_J[j])*P_symbol((d-j)/2,alpha_top_J_plus1[j])*(alpha_J[j]+lambda_J[j])/(P_symbol(2*lambda_J[j],alpha_J[j])*P_symbol((d-j-1)/2,alpha_top_J_plus1[j])*lambda_J[j])
        List.append(np.sqrt(2*S))
    L.append(np.array(List[:]))
'''


d =785
H_alpha = computation_h_alpha(d)

H_alpha[2][0]
   
        

In [ ]:
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
transform = transforms.Compose([transforms.ToTensor()])
trainset_tmp = datasets.MNIST('~/', download=True, train=True, transform=transform)
trainloader_tmp = torch.utils.data.DataLoader(trainset_tmp, batch_size=10_000, shuffle=True)
testset = datasets.MNIST('~/', download=True, train=False, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=512, shuffle=False)

In [ ]:
def data_divides__minst():
    Train = []
    for image in trainset_tmp:
        Ima = image[0][0].reshape(1,784)
        if image[1]%2 == 0:
            number = 1
        elif image[1]%2 == 1:
            number = -1
        Train.append([np.array(Ima[0][:]),number])
    return Train
Train = data_divides__minst()

def data_divides__minst_test():
    Test = []
    for image in testset:
        Ima = image[0][0].reshape(1,784)
        if image[1]%2 == 0:
            number = 1
        elif image[1]%2 == 1:
            number = -1
        Test.append([np.array(Ima[0][:]),number])
    return Test
Test = data_divides__minst_test()
#训练集测试集合处理为球面数据
Test_data = []
for image in Test:
    image_pic = image[0]/10
    a = 2/(sum(image_pic*image_pic)+1)
    Ima = np.append(image_pic*a, a-1)
    Test_data.append([Ima[:],image[1]])

In [ ]:
'has projection'
import numpy as np
import math
import random
import time
class T_kernel_SGD():
    def __init__(self, d, gamma_0, theta, t, s,N,H_alpha,Q):
        self.d = d
        self.s = s
        self.gamma_0 = gamma_0
        self.theta = theta
        self.t = t
        self.f_hat = [np.array([0])]
        self.f_bar = [np.array([0])]
        self.sgd_f = [0 for k in range(self.d)]
        self.Z_0 = np.array(1)
        self.Z_1 = self.d*np.ones(self.d)
        self.Z_2 = [-(self.d+2)/2,np.zeros(self.d),np.ones((self.d,self.d))*(self.d*(self.d+2))/2]
        self.Znt = [self.Z_0]
        self.ZntX = []
        self.order = 0
        self.dimPiLn = 1
        self.N = N
        self.H_alpha = H_alpha
        self.Q = Q

    def C_n_lambda(self, lambda_n, nn, x):
        if nn == 1:
            S = 2*lambda_n*x
        elif nn == 2:
            S = 2*lambda_n*(1+lambda_n)*(x**2)-lambda_n
        return S
        
    def computation_Y_KJ(self, X):#(X_d,\dots,X_1),X need to be numpy.array list
        self.Y_KJ_List = []
        if self.order == 0:
            self.Y_KJ_List = [np.array([1])]
        elif self.order == 1:
            self.Y_KJ_List = [np.array([1])]
            self.Y_KJ_List.append(self.H_alpha[1][0]*X)
        elif self.order == 2:
            self.Y_KJ_List = [np.array([1])]
            self.Y_KJ_List.append(self.H_alpha[1][0]*X)
            X_inv = np.flip(X)
            Sum_X_list = []
            sec_Y_KJ_List = []
            S = 0
            for i in X_inv:
                S += i**2
                Sum_X_list.insert(0,S)
            X_turncated = self.H_alpha[2][0][-1]*X[:]
            for k in range(1,d-1):
                Coe_List = X[k-1]*X_turncated
                Coe_List[0] = Sum_X_list[k-1]*self.C_n_lambda((self.d-k-1)/2, 2, X[k-1]/np.sqrt(Sum_X_list[k-1]))*self.H_alpha[2][k-1][0]
                sec_Y_KJ_List.append(Coe_List[:])
                X_turncated = X_turncated[1:]
            sec_Y_KJ_List.append(np.array([X_inv[0]**2-X_inv[1]**2,2*X_inv[0]*X_inv[1]])*self.H_alpha[2][self.d-2])
            self.Y_KJ_List.append(sec_Y_KJ_List[:])
        
            
    def comput_f_bar(self,X):
        self.computation_Y_KJ(X)
        if self.order == 0:
            y = self.f_bar[0][0]
        elif self.order == 1:
            y = self.f_bar[0][0]+np.dot(self.f_bar[1],self.Y_KJ_List[1])
        elif self.order == 2:
            y = self.f_bar[0][0]+np.dot(self.f_bar[1],self.Y_KJ_List[1])+sum([ np.dot(a,b) for a,b in zip(self.f_bar[2],self.Y_KJ_List[2])])
        return y
    
    def comput_f_hat(self,X):
        self.computation_Y_KJ(X)
        if self.order == 0:
            y = self.f_hat[0][0]
        elif self.order == 1:
            y = self.f_hat[0][0]+np.dot(self.f_hat[1],self.Y_KJ_List[1])
        elif self.order == 2:
            y = self.f_hat[0][0]+np.dot(self.f_hat[1],self.Y_KJ_List[1])+sum([ np.dot(a,b) for a,b in zip(self.f_hat[2],self.Y_KJ_List[2])])
        return y
    
    def add_gaussian_noise_np(self, np_array, mean=0, std=1):
        noise = np.random.normal(mean, std, np_array.shape)
        self.data = (np_array + noise)/10
        
    def train_generate_data(self):
        DATA = random.sample(Train,1)
        self.add_gaussian_noise_np(DATA[0][0], mean = 0, std = 0.0005)
        a = 2/(sum(self.data*self.data)+1)
        Ima = np.append(self.data*a,a-1)
        return [Ima[:],DATA[0][1]]

        
    def algorithm(self,X,Y):
        gamma_n = self.gamma_0*(self.epoch**(-self.t))
        #下面是导数，对于图像分类要换掉
        z = np.exp(Y*self.comput_f_hat(X))
        coe = gamma_n*Y/(1+z)
        #F = self.comput_F_hat(X)
        #COE = gamma_n*(Y-F)
        if self.epoch**self.theta >= self.dimPiLn:
            self.order = self.order + 1
            if self.order == 1:
                self.dimPiLn = self.d + 1
                self.f_hat.append(np.zeros(self.d))
                self.f_bar.append(np.zeros(self.d))
            elif self.order == 2:
                self.dimPiLn = (self.d+3)*self.d/2
                self.f_hat.append([ np.zeros(self.d-j) for j in range(self.d-1)])
                self.f_bar.append([ np.zeros(self.d-j) for j in range(self.d-1)])
        self.computation_Y_KJ(X)
        self.f_hat[0] = self.f_hat[0] + coe*self.Y_KJ_List[0]
        if self.order >= 1:
            self.f_hat[1] = self.f_hat[1] + coe*((self.d + 1)**(-2*self.s))*self.Y_KJ_List[1]
        if self.order >= 2:
            coe1 = coe*(((self.d+3)*self.d/2)**(-2*self.s))
            self.f_hat[2] = [ a + coe1*b for a,b in zip(self.f_hat[2],self.Y_KJ_List[2]) ]
        if self.order == 0:
            self.Norm = np.sqrt(self.f_hat[0][0]**2)
        elif self.order == 1:
            self.Norm = np.sqrt(self.f_hat[0][0]**2 + (self.d + 1)**(2*self.s)*np.sum(self.f_hat[1]**2))
        elif self.order == 2:
            self.Norm = np.sqrt(self.f_hat[0][0]**2 + (self.d + 1)**(2*self.s)*np.sum(self.f_hat[1]**2) + ((self.d+3)*self.d/2)**(2*self.s)*sum([ np.sum(a**2) for a in self.f_hat[2]]))
        if self.Norm >= self.Q:
            zoom = self.Q/self.Norm
            self.f_hat[0] = zoom*self.f_hat[0]
            if self.order >= 1:
                self.f_hat[1] = zoom*self.f_hat[1]
            if self.order >= 2:
                self.f_hat[2] = [ zoom*a for a in self.f_hat[2] ]
        self.f_bar[0] =  (self.f_hat[0] + self.epoch*self.f_bar[0])/(self.epoch+1)
        if self.order >= 1:
            self.f_bar[1] =  (self.f_hat[1] + self.epoch*self.f_bar[1])/(self.epoch+1)
        if self.order >= 2:
            self.f_bar[2] = [ (a+self.epoch*b)/(self.epoch +1) for a,b in zip(self.f_hat[2],self.f_bar[2]) ]
        #error = np.log(1+np.exp(z))
        return 0#error 

    def test(self,T):
        Error_last = 0
        Error_ave = 0
        Accac_last = 0
        Accac_ave = 0
        for k in range(T):
            Z = Test_data[k]
            #这里误差衡量也要改
            f_hat = self.comput_f_hat(Z[0])
            f_ave = self.comput_f_bar(Z[0])
            e_ave = np.square(f_ave-Z[1])
            e_last = np.square(f_hat-Z[1])
            Error_ave += e_ave
            Error_last += e_last
            if f_ave*Z[1] > 0:
                Accac_ave += 1
            if f_hat*Z[1] > 0:
                Accac_last += 1
        self.ERROR_last.append(Error_last/T)
        self.ERROR_ave.append(Error_ave/T)
        self.Accuracy_ave.append(Accac_ave/T)
        self.Accuracy_last.append(Accac_last/T)
        print(self.epoch,self.ERROR_last[-1], self.ERROR_ave[-1],self.Accuracy_last[-1],self.Accuracy_ave[-1])
        
    def update(self,n,operation):
        self.Accuracy_last = []
        self.Accuracy_ave = []
        self.ERROR_last = []
        self.ERROR_ave = []
        self.Time = []
        start = time.perf_counter()
        T = len(Test_data) #测试数据的多少
        for i in range(1,n+1):
            self.epoch = i
            DATA = self.train_generate_data()
            error = self.algorithm(DATA[0],DATA[1])
            '''if error < np.log(2):
                Acc += 1
            Error += error'''
            if self.epoch in self.N:
                if operation == 'time':
                    end = time.perf_counter()
                    runTime = end - start
                    self.Time.append(runTime)
                    print(self.epoch, runTime)
                elif operation == 'test':    
                    print(self.epoch,'Norm', self.Norm)
                    self.test(T)
        return [self.ERROR_last,self.ERROR_ave,self.Accuracy_last,self.Accuracy_ave]

In [ ]:
d = 785
gamma_0 = 0.6
theta = 0.68
t = 0.05
s = 0.505
Q = 1000
N = [int(10**(i/5))for i in range(1,36)]
for i in range(1):
    t_kernel_SGD = T_kernel_SGD(d, gamma_0, theta, t, s,N,H_alpha, Q)
    B = t_kernel_SGD.update(1600004,'test')

In [ ]:
d = 785
gamma_0 = 0.6
theta = 0.68
t = 0.05
s = 0.505
Q = 1000
N = [int(10**(i/5))for i in range(1,36)]
for i in range(1):
    t_kernel_SGD = T_kernel_SGD(d, gamma_0, theta, t, s,N,H_alpha, Q)
    B = t_kernel_SGD.update(1600004,'time')